# **Proyecto Etapa 4 — Aprendizaje No Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

**Actividad individual**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A01139580 |

---
## 1. Introducción: Aprendizaje No Supervisado

### 1.1 Concepto general

El **aprendizaje no supervisado** es una rama del aprendizaje automático en la que el modelo no dispone de etiquetas $(y_i)$ durante el entrenamiento. El objetivo es descubrir **estructura latente** en los datos: ya sea agrupando instancias similares (*clustering*), reduciendo su dimensionalidad o modelando la distribución subyacente.

Dado un conjunto sin etiquetas $\mathcal{D} = \{x_1, x_2, \ldots, x_n\}$, el problema de clustering particional busca una asignación $C: \{1,\ldots,n\} \to \{1,\ldots,k\}$ que minimice la distancia intra-cluster. Para K-Means, el objetivo es:

$$\min_{\mu_1,\ldots,\mu_k} \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

Las principales tareas del aprendizaje no supervisado son:

- **Clustering**: agrupar instancias similares sin etiquetas (K-Means, GMM, DBSCAN, clustering jerárquico).
- **Reducción de dimensionalidad**: proyectar en un espacio de menor dimensión preservando la varianza o la estructura local (PCA, t-SNE, UMAP, autoencoders).
- **Modelado generativo**: aprender la distribución de los datos para generar nuevas muestras (VAE, GAN).
- **Detección de anomalías**: identificar instancias que se desvían de la distribución normal aprendida.

---

### 1.2 Algoritmos representativos en la literatura

| Algoritmo | Tipo | Fortalezas | Limitaciones |
|-----------|------|------------|--------------|
| **K-Means** | Clustering particional | Simple, eficiente, escalable con grandes volúmenes | Asume clusters esféricos e isotrópicos; sensible a escala y outliers |
| **K-Means++** | Clustering particional | Inicialización mejorada → convergencia más estable y resultados de mejor calidad | Mismo costo computacional que K-Means; limitaciones geométricas iguales |
| **Bisecting K-Means** | Clustering jerárquico | Produce jerarquía de clusters; menos sensible a inicialización | Menos flexible en forma de clusters; parámetro k sigue siendo necesario |
| **GMM (Gaussian Mixture Model)** | Clustering probabilístico | Asignación suave (*soft*); permite clusters elípticos; modelado probabilístico | Más costoso (algoritmo EM); puede diverger; supone distribución gaussiana |
| **DBSCAN** | Clustering por densidad | Detecta outliers de forma natural; no requiere especificar k | Sensible a parámetros `eps` y `minPts`; difícil de escalar en alta dimensión |
| **Agglomerative Clustering** | Clustering jerárquico | Produce dendrograma; no requiere k a priori | $O(n^2)$ en memoria y tiempo; difícil de escalar con grandes conjuntos de datos |
| **PCA** | Reducción de dimensionalidad | Lineal, eficiente, interpretable (varianza explicada) | Solo captura relaciones lineales; componentes no necesariamente interpretables |
| **LDA (Latent Dirichlet Allocation)** | Modelado de tópicos | Interpretable para datos de texto y conteos genómicos | Específico para distribuciones discretas |

---

### 1.3 Implementaciones disponibles en PySpark MLlib

PySpark MLlib (`pyspark.ml.clustering`) ofrece las siguientes implementaciones distribuidas:

| Clase PySpark | Algoritmo | Descripción |
|---------------|-----------|-------------|
| `KMeans` | K-Means (K-Means++) | Clustering particional estándar con inicialización K-Means++; soporta distancia euclidiana y coseno |
| `BisectingKMeans` | Bisecting K-Means | Variante divisiva jerárquica; divide iterativamente el cluster con mayor WSSSE |
| `GaussianMixture` | GMM | Mezcla de gaussianas; asignación probabilística (*soft clustering*) mediante el algoritmo EM |
| `LDA` | Latent Dirichlet Allocation | Modelado de tópicos para texto y datos de conteo; variantes online y EM distribuidas |
| `PowerIterationClustering` | PIC | Clustering espectral escalable para grafos de similitud; basado en eigenvectores aproximados |

Para reducción de dimensionalidad, PySpark ofrece `PCA` en `pyspark.ml.feature`.

La calidad del clustering se evalúa con `ClusteringEvaluator`, que implementa el **índice Silhouette** — la métrica estándar para medir cohesión intra-cluster y separación inter-cluster:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, 1]$$

donde $a(i)$ es la distancia media intra-cluster y $b(i)$ es la distancia media al cluster vecino más cercano.

---

### 1.4 Algoritmo seleccionado: K-Means

Se selecciona **K-Means** (con inicialización K-Means++) como algoritmo principal, por las siguientes razones en el contexto GTEx:

1. **Escalabilidad**: K-Means en PySpark utiliza un algoritmo mini-batch distribuido, manejando eficientemente las 799 muestras × 500 features.
2. **Interpretabilidad**: los centroides representan perfiles de expresión génica "promedio" para cada cluster, permitiendo interpretación biológica directa.
3. **Baseline estándar**: K-Means es el algoritmo de clustering de referencia en bioinformática para RNA-seq.
4. **Compatibilidad con PCA**: la combinación `StandardScaler → PCA → K-Means` es el pipeline estándar para datos de alta dimensionalidad genómica.

Como experimento complementario se aplica **GMM** (`GaussianMixture`) para comparar los resultados con asignaciones probabilísticas y evaluar si los clusters de expresión génica siguen distribuciones gaussianas en el espacio PCA.

**Hipótesis**: dado que en la Tarea 3 se demostró que los perfiles de expresión cardiovascular y musculoesquelético son perfectamente separables con aprendizaje supervisado, se espera que K-Means *descubra* estos mismos grupos sin usar etiquetas, obteniendo un índice Silhouette alto y clusters que se alineen con los grupos de tejido conocidos.

In [1]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
import pandas as pd
from collections import defaultdict
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sub-sample parameters (same as Tarea 3 for comparability)
TARGET_TISSUE_GROUPS  = ['Cardiovascular', 'Musculoesqueletico']
SAMPLES_PER_PARTITION = 200   # max samples per (tissue_group × sex) partition
N_GENES_VARIANCE      = 500   # top-N genes by inter-sample variance
N_PCA_COMPONENTS      = 50    # PCA dimensions before K-Means

print(f'Semilla aleatoria    : {RANDOM_SEED}')
print(f'Grupos de tejido     : {TARGET_TISSUE_GROUPS}')
print(f'Muestras/partición   : {SAMPLES_PER_PARTITION}')
print(f'Genes (por varianza) : {N_GENES_VARIANCE}')
print(f'Componentes PCA      : {N_PCA_COMPONENTS}')
print(f'JAVA_HOME            : {os.environ.get("JAVA_HOME", "ERROR - no seteado")}')

Semilla aleatoria    : 42
Grupos de tejido     : ['Cardiovascular', 'Musculoesqueletico']
Muestras/partición   : 200
Genes (por varianza) : 500
Componentes PCA      : 50
JAVA_HOME            : C:\Users\diego\anaconda3\envs\big-data\Library\lib\jvm


In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_UnsupervisedLearning_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark

---
## 2. Selección de los datos

### 2.1 Estrategia

Se construye la sub-muestra **M'** a partir de la muestra de equipo M (Etapa 2), siguiendo la misma estrategia de la Tarea 3 para mantener comparabilidad:

- Se trabaja con las **particiones cardiovasculares y musculoesqueléticas**, que corresponden a los grupos de tejido de mayor interés biológico en el contexto del proyecto.
- Se aplica un muestreo **estratificado proporcional por subtipo de tejido (SMTSD)** dentro de cada partición (TISSUE_GROUP × SEX_LABEL), tomando hasta **200 muestras por partición**.
- Resultado esperado: ~799 muestras (~400 cardiovasculares + ~400 musculoesqueléticas).
- Las **etiquetas de tejido** se guardan por separado y **NO se utilizan durante el entrenamiento** (escenario no supervisado). Solo se usan en la validación externa post-clustering.

**Selección de genes**: se seleccionan los **500 genes de mayor varianza** entre las 799 muestras. Esta selección captura genes diferencialmente expresados entre tejidos sin usar información de las etiquetas, siendo apropiada para el contexto no supervisado.

**Justificación del tamaño de M'**: 800 muestras × 500 genes da una matriz manejable para PySpark en modo local, y suficiente para que K-Means produzca clusters estadísticamente estables.

In [3]:
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

meta_target = meta_df.filter(F.col('TISSUE_GROUP').isin(TARGET_TISSUE_GROUPS))

print('Distribución por partición (grupos objetivo):')
meta_target.groupBy('TISSUE_GROUP', 'SEX_LABEL').count().orderBy('TISSUE_GROUP', 'SEX_LABEL').show()

Distribución por partición (grupos objetivo):


+------------------+---------+-----+
|      TISSUE_GROUP|SEX_LABEL|count|
+------------------+---------+-----+
|    Cardiovascular| Femenino|  771|
|    Cardiovascular|Masculino| 1573|
|Musculoesqueletico| Femenino| 1349|
|Musculoesqueletico|Masculino| 2827|
+------------------+---------+-----+



In [4]:
meta_rows = meta_target.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD').collect()

partition_samples = defaultdict(list)
for row in meta_rows:
    key = (row['TISSUE_GROUP'], row['SEX_LABEL'])
    partition_samples[key].append((row['COL_NAME'], row['SMTSD']))

rng = random.Random(RANDOM_SEED)
selected_col_names = []
selected_meta = []   # (col_name, tissue_group, sex_label)

for (tg, sx), items in sorted(partition_samples.items()):
    n_partition = len(items)
    n_select = min(SAMPLES_PER_PARTITION, n_partition)

    by_subtype = defaultdict(list)
    for col, smtsd in items:
        by_subtype[smtsd].append(col)

    sampled = []
    for smtsd, cols in by_subtype.items():
        n_strata = max(1, round(n_select * len(cols) / n_partition))
        n_strata = min(n_strata, len(cols))
        sampled.extend(rng.sample(cols, n_strata))

    sampled = sampled[:n_select]

    for col in sampled:
        selected_col_names.append(col)
        selected_meta.append((col, tg, sx))

    print(f'  {tg:<22} + {sx:<12}: {len(sampled)} muestras de {n_partition}')

# Build label lookup — stored separately, NOT used during training
meta_dict = {col: tg for col, tg, sx in selected_meta}
print(f'\nTotal M\' : {len(selected_col_names)} muestras')

  Cardiovascular         + Femenino    : 200 muestras de 771
  Cardiovascular         + Masculino   : 199 muestras de 1573
  Musculoesqueletico     + Femenino    : 200 muestras de 1349
  Musculoesqueletico     + Masculino   : 200 muestras de 2827

Total M' : 799 muestras


In [5]:
from pyspark.sql.functions import split as spark_split

peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names_raw   = peek.columns.tolist()
all_col_names_clean = [c.replace('-', '_').replace('.', '_') for c in all_col_names_raw]
name_to_idx = {clean: idx for idx, clean in enumerate(all_col_names_clean)}

fixed_cols    = ['Name', 'Description']
fixed_indices = [name_to_idx[c] for c in fixed_cols]

valid_sample_cols = [c for c in selected_col_names if c in name_to_idx]
sample_indices    = [name_to_idx[c] for c in valid_sample_cols]

all_selected_indices = fixed_indices + sample_indices
all_selected_names   = fixed_cols + valid_sample_cols

raw_df  = spark.read.text(FILE_PATH)
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))
split_col = spark_split(F.col('value'), '\t')

df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(all_selected_names[idx])
      for idx, i in enumerate(all_selected_indices)]
)

for c in valid_sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM M\': {df_tpm.count():,} genes × {len(valid_sample_cols)} muestras')
df_tpm.select(all_selected_names[:5]).show(3)

DataFrame TPM M': 59,033 genes × 799 muestras


+-----------------+-----------+-----------------------+------------------------+------------------------+
|             Name|Description|GTEX_QDT8_0426_SM_32PKZ|GTEX_13CF3_2226_SM_5J2MX|GTEX_11GSP_2926_SM_5N9C2|
+-----------------+-----------+-----------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                    0.0|                     0.0|                     0.0|
|ENSG00000227232.5|     WASH7P|                5.78407|                 3.13248|                 3.17456|
|ENSG00000278267.1|  MIR6859-1|                    0.0|                     0.0|                     0.0|
+-----------------+-----------+-----------------------+------------------------+------------------------+
only showing top 3 rows


In [6]:
print('Calculando varianza inter-muestras para todos los genes...')
tpm_full_pd = df_tpm.select(['Name'] + valid_sample_cols).toPandas()
tpm_full_pd = tpm_full_pd.set_index('Name')

gene_var    = tpm_full_pd.var(axis=1).sort_values(ascending=False)
top_var_ids = gene_var.head(N_GENES_VARIANCE).index.tolist()

desc_map = df_tpm.select('Name', 'Description').toPandas() \
    .set_index('Name')['Description'].to_dict()

print(f'\nTop 10 genes por varianza (candidatos a marcadores tisulares):')
print(f'  {"Rank":<5} {"Ensembl ID":<30} {"Símbolo":<15} {"Varianza":>18}')
print('  ' + '-' * 72)
for rank, eid in enumerate(top_var_ids[:10], 1):
    sym = desc_map.get(eid, '?')
    print(f'  {rank:<5} {eid:<30} {sym:<15} {gene_var[eid]:>18,.0f}')

Calculando varianza inter-muestras para todos los genes...



Top 10 genes por varianza (candidatos a marcadores tisulares):
  Rank  Ensembl ID                     Símbolo                   Varianza
  ------------------------------------------------------------------------
  1     ENSG00000198804.2              MT-CO1                 566,183,744
  2     ENSG00000198938.2              MT-CO3                 430,945,984
  3     ENSG00000198899.2              MT-ATP6                406,831,616
  4     ENSG00000198886.2              MT-ND4                 394,051,104
  5     ENSG00000198712.1              MT-CO2                 296,982,080
  6     ENSG00000210082.2              MT-RNR2                195,647,664
  7     ENSG00000198888.2              MT-ND1                 173,899,232
  8     ENSG00000198727.2              MT-CYB                 166,914,752
  9     ENSG00000175206.11             NPPA                   163,619,712
  10    ENSG00000198763.3              MT-ND2                 139,426,624


In [7]:
valid_genes = [g for g in top_var_ids if g in tpm_full_pd.index]
gene_cols   = [g.replace('.', '_') for g in valid_genes]

tpm_T = tpm_full_pd.loc[valid_genes].T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})
tpm_T = tpm_T.rename(columns={g: g.replace('.', '_') for g in valid_genes})
tpm_T[gene_cols] = tpm_T[gene_cols].fillna(0.0)

# Etiquetas guardadas como columna auxiliar — NO formarán parte del entrenamiento
tpm_T['TISSUE_GROUP'] = tpm_T['COL_NAME'].map(meta_dict)
tpm_T = tpm_T.dropna(subset=['TISSUE_GROUP'])

print(f'Matriz M\' final : {tpm_T.shape[0]} muestras × {len(gene_cols)} genes')
print('Distribución por grupo de tejido:')
print(tpm_T['TISSUE_GROUP'].value_counts().to_dict())

Matriz M' final : 799 muestras × 500 genes
Distribución por grupo de tejido:
{'Musculoesqueletico': 400, 'Cardiovascular': 399}


C:\Users\diego\AppData\Local\Temp\ipykernel_7444\1092121133.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tpm_T['TISSUE_GROUP'] = tpm_T['COL_NAME'].map(meta_dict)


---
## 3. Preparación del conjunto de entrenamiento y prueba

### 3.1 Técnica de división

Se aplica una **división estratificada 80 / 20** (train / test):

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Train | 80% (~639 muestras) | Proporciona suficientes datos para que K-Means y GMM ajusten centroides estables en 50 dimensiones PCA |
| Test | 20% (~160 muestras) | Conjunto independiente para calcular el Silhouette sobre instancias no vistas durante el entrenamiento |
| Estratificación | Por `TISSUE_GROUP` | Mantiene la misma proporción de clases en ambos subconjuntos, evitando que el test tenga distribución distinta |
| Semilla | `RANDOM_SEED = 42` | Reproducibilidad de resultados |

**Nota sobre aprendizaje no supervisado**: a diferencia del aprendizaje supervisado, las etiquetas (`TISSUE_GROUP`) **no se utilizan durante el entrenamiento**. La división estratificada se aplica únicamente para asegurar que el conjunto de prueba tenga representación balanceada de los grupos, permitiendo una evaluación externa (post-hoc) más robusta.

El pipeline de preprocesamiento (`StandardScaler`, `PCA`) se ajusta **exclusivamente sobre el conjunto de entrenamiento** y se aplica al de prueba, siguiendo el principio de no fuga de información (*data leakage prevention*).

In [8]:
from sklearn.model_selection import train_test_split

indices  = list(range(len(tpm_T)))
y_groups = tpm_T['TISSUE_GROUP'].values

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_groups
)

train_df = tpm_T.iloc[train_idx].reset_index(drop=True)
test_df  = tpm_T.iloc[test_idx].reset_index(drop=True)

print(f'Entrenamiento : {len(train_df)} muestras  ({len(train_df)/len(tpm_T)*100:.1f}%)')
print(f'Prueba        : {len(test_df)} muestras  ({len(test_df)/len(tpm_T)*100:.1f}%)')
print(f'\nGrupos en train: {dict(train_df["TISSUE_GROUP"].value_counts())}')
print(f'Grupos en test : {dict(test_df["TISSUE_GROUP"].value_counts())}')

Entrenamiento : 639 muestras  (80.0%)
Prueba        : 160 muestras  (20.0%)

Grupos en train: {'Musculoesqueletico': np.int64(320), 'Cardiovascular': np.int64(319)}
Grupos en test : {'Musculoesqueletico': np.int64(80), 'Cardiovascular': np.int64(80)}


In [9]:
# Incluir TISSUE_GROUP y COL_NAME en el DataFrame de Spark solo como
# columnas auxiliares — el VectorAssembler las ignorará durante el clustering
train_spark = spark.createDataFrame(train_df[gene_cols + ['COL_NAME', 'TISSUE_GROUP']])
test_spark  = spark.createDataFrame(test_df[gene_cols  + ['COL_NAME', 'TISSUE_GROUP']])

print(f'Spark train : {train_spark.count()} filas')
print(f'Spark test  : {test_spark.count()} filas')

Spark train : 639 filas


Spark test  : 160 filas


---
## 4. Construcción de modelos de aprendizaje no supervisado

### 4.1 Preprocesamiento de features

Antes de aplicar K-Means o GMM se requieren dos etapas de preparación:

1. **`StandardScaler`** (`with_mean=True, with_std=True`): centra y normaliza cada gen a media 0 y desviación estándar 1. K-Means es sensible a la escala.

2. **`PCA`** (`n_components=50`): reduce la dimensionalidad de 500 a 50 componentes principales, aliviando la maldición de la dimensionalidad y acelerando K-Means.

**Pipeline de implementación:**

| Paso | Herramienta | Razón |
|------|-------------|-------|
| StandardScaler + PCA | scikit-learn | Opera sobre pandas en memoria; sin problemas de tipos en Spark |
| VectorAssembler | PySpark MLlib | Convierte columnas numéricas a vector denso compatible con MLlib |
| KMeans, GaussianMixture | PySpark MLlib | Entrenamiento y predicción distribuida |
| Silhouette | scikit-learn post-hoc | Evaluación sobre predicciones recopiladas del cluster |

El mismo patrón mixto (sklearn para preprocesamiento, Spark para el modelo) se usó en la Tarea 3 (sklearn para train/test split, luego Spark para Random Forest). Los algoritmos de clustering se ejecutan **íntegramente en PySpark MLlib**.

El pipeline se ajusta **únicamente sobre el conjunto de entrenamiento** (no data leakage).

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA
from pyspark.ml.feature import VectorAssembler

# --- Preprocessing (sklearn, fitted only on train) ---
X_train = train_df[gene_cols].values.astype(float)
X_test  = test_df[gene_cols].values.astype(float)

scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

pca_sk      = SklearnPCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_SEED)
X_train_pca = pca_sk.fit_transform(X_train_scaled)
X_test_pca  = pca_sk.transform(X_test_scaled)

cumul_var = np.cumsum(pca_sk.explained_variance_ratio_)
print('Varianza explicada acumulada:')
print(f'  PC 1-5  : {cumul_var[4]*100:.1f}%')
print(f'  PC 1-10 : {cumul_var[9]*100:.1f}%')
print(f'  PC 1-50 : {cumul_var[-1]*100:.1f}%')

# --- Build pandas DataFrames with individual PC columns ---
pca_cols = [f'pc{i:02d}' for i in range(N_PCA_COMPONENTS)]

def make_pca_df(X_pca, df_meta):
    d = pd.DataFrame(X_pca, columns=pca_cols)
    d['TISSUE_GROUP'] = df_meta['TISSUE_GROUP'].values
    d['COL_NAME']     = df_meta['COL_NAME'].values
    return d

# --- Convert to Spark and assemble feature vector with VectorAssembler ---
# VectorAssembler is the standard MLlib way to build feature vectors from numeric columns
assembler_pca = VectorAssembler(inputCols=pca_cols, outputCol='pca_features')

train_pca = assembler_pca.transform(spark.createDataFrame(make_pca_df(X_train_pca, train_df))) \
    .select('pca_features', 'TISSUE_GROUP', 'COL_NAME').cache()
test_pca  = assembler_pca.transform(spark.createDataFrame(make_pca_df(X_test_pca,  test_df))) \
    .select('pca_features', 'TISSUE_GROUP', 'COL_NAME').cache()

n_train = train_pca.count()
n_test  = test_pca.count()
print(f'\ntrain_pca : {n_train} filas  |  test_pca : {n_test} filas')
first_vec = train_pca.first()['pca_features']
print(f'pca_features dtype: {type(first_vec).__name__}  |  dim: {len(first_vec)}')
print(f'PC00-04 del primer vector: {list(first_vec[:5])}')

Varianza explicada acumulada:
  PC 1-5  : 62.4%
  PC 1-10 : 72.3%
  PC 1-50 : 90.7%



train_pca : 639 filas  |  test_pca : 160 filas
pca_features dtype: DenseVector  |  dim: 50
PC00-04 del primer vector: [np.float64(-14.103173832665691), np.float64(-2.2021744404767576), np.float64(-1.2603107773093192), np.float64(-6.6931754101145), np.float64(-3.6600845115996457)]


### 4.2 Método del codo: selección de k para K-Means

Para determinar el número óptimo de clusters $k$ se evalúa el **índice Silhouette** en el conjunto de prueba para $k \in \{2, 3, 4, 5, 6\}$.

El índice Silhouette mide simultáneamente la **cohesión** intra-cluster y la **separación** inter-cluster:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, 1]$$

- **> 0.70**: clusters compactos y bien separados.
- **0.50–0.70**: estructura razonable.
- **0.25–0.50**: débil — podría ser artificial.
- **< 0.25**: sin estructura clara.

El Silhouette se calcula post-hoc con **sklearn** sobre las predicciones recopiladas de Spark (misma métrica, diferente implementación que `ClusteringEvaluator`). Se elige el $k$ que maximiza el Silhouette medio.

In [11]:
from pyspark.ml.clustering import KMeans
from sklearn.metrics import silhouette_score as sk_silhouette

print('Método del codo: Silhouette (test set, sklearn post-hoc)')
print(f'  {"k":<5} {"Silhouette":>12} {"WCSS train":>14}')
print('  ' + '-' * 34)

silhouette_scores = {}
for k in range(2, 7):
    km = KMeans(
        featuresCol='pca_features', predictionCol='prediction',
        k=k, maxIter=30, seed=RANDOM_SEED
    )
    km_model = km.fit(train_pca)
    wssse    = km_model.summary.trainingCost

    preds_pd = km_model.transform(test_pca).select('pca_features', 'prediction').toPandas()
    y_pred   = preds_pd['prediction'].values
    X_eval   = np.vstack([v.toArray() for v in preds_pd['pca_features']])

    if len(np.unique(y_pred)) < 2:
        print(f'  {k:<5} {"(degenerate)":>12} {wssse:>14,.1f}')
        silhouette_scores[k] = -1.0
        continue

    score = sk_silhouette(X_eval, y_pred, metric='euclidean')
    silhouette_scores[k] = score
    print(f'  {k:<5} {score:>12.4f} {wssse:>14,.1f}')

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f'\nMejor k según Silhouette: k={best_k}  (score={silhouette_scores[best_k]:.4f})')

Método del codo: Silhouette (test set, sklearn post-hoc)
  k       Silhouette     WCSS train
  ----------------------------------


  2           0.1778      254,286.8


  3           0.2785      176,129.4


  4           0.3129      148,688.6


  5           0.3688      129,283.0


  6           0.2671      125,132.0

Mejor k según Silhouette: k=5  (score=0.3688)


### 4.3 K-Means con k=2

Se entrena el modelo final con $k=2$, que corresponde al número de grupos de tejido conocidos (Cardiovascular y Musculoesquelético). Esto permite comparar directamente los clusters descubiertos por el algoritmo con los grupos reales como validación externa.

**Supuestos del modelo:**
1. Los clusters tienen forma aproximadamente esférica en el espacio PCA.
2. Los puntos de datos se distribuyen de manera relativamente uniforme dentro de cada cluster.
3. La varianza de los genes seleccionados captura las principales diferencias entre tejidos.
4. La normalización (StandardScaler) elimina el efecto de la escala absoluta de expresión.

In [12]:
K_FINAL = 2  # número de grupos de tejido esperados

kmeans = KMeans(
    featuresCol='pca_features',
    predictionCol='prediction',
    k=K_FINAL,
    maxIter=30,
    seed=RANDOM_SEED
)

print(f'Entrenando K-Means (k={K_FINAL}, PCA {N_PCA_COMPONENTS} componentes, maxIter=30)...')
model_km = kmeans.fit(train_pca)
print('Entrenamiento completado.')
print(f'\nCosto de entrenamiento (WCSS): {model_km.summary.trainingCost:,.2f}')
print(f'Iteraciones hasta convergencia: {model_km.summary.numIter}')

Entrenando K-Means (k=2, PCA 50 componentes, maxIter=30)...


Entrenamiento completado.

Costo de entrenamiento (WCSS): 254,286.75
Iteraciones hasta convergencia: 6


In [13]:
from sklearn.metrics import silhouette_score as sk_silhouette

preds_km = model_km.transform(test_pca)

# Collect predictions for evaluation and external validation
preds_km_pd = preds_km.select('pca_features', 'prediction', 'TISSUE_GROUP').toPandas()
y_pred_km   = preds_km_pd['prediction'].values
X_eval_km   = np.vstack([v.toArray() for v in preds_km_pd['pca_features']])

print('Distribución de clusters K-Means en test:')
unique_km, counts_km = np.unique(y_pred_km, return_counts=True)
for cl, cnt in zip(unique_km, counts_km):
    print(f'  Cluster {cl}: {cnt} muestras')

if len(unique_km) >= 2:
    sil_km = sk_silhouette(X_eval_km, y_pred_km, metric='euclidean')
else:
    sil_km = -1.0
    print('AVISO: clustering degenerado — Silhouette indefinido')

print(f'\nSilhouette score (K-Means, k={K_FINAL}): {sil_km:.4f}')
print('\nEscala de referencia:')
print('  > 0.70  : estructura fuerte')
print('  0.50-0.70: estructura razonable')
print('  0.25-0.50: débil')
print('  < 0.25  : sin estructura clara')

Distribución de clusters K-Means en test:
  Cluster 0: 55 muestras
  Cluster 1: 105 muestras

Silhouette score (K-Means, k=2): 0.1778

Escala de referencia:
  > 0.70  : estructura fuerte
  0.50-0.70: estructura razonable
  0.25-0.50: débil
  < 0.25  : sin estructura clara


In [14]:
# Validación externa post-hoc: ¿los clusters coinciden con los grupos de tejido?
# Las etiquetas NUNCA se usaron durante el entrenamiento
print('Validación externa: distribución de grupos de tejido por cluster K-Means\n')
preds_km.groupBy('prediction', 'TISSUE_GROUP').count() \
    .orderBy('prediction', 'TISSUE_GROUP') \
    .show()

contingency = pd.crosstab(
    preds_km_pd['prediction'],
    preds_km_pd['TISSUE_GROUP'],
    margins=True, margins_name='Total'
)
print('Tabla de contingencia (cluster × grupo de tejido):')
print(contingency)

if len(unique_km) >= 2:
    purity = sum(contingency.iloc[i].drop('Total').max() for i in range(K_FINAL)) / len(preds_km_pd)
    print(f'\nPureza del clustering: {purity:.4f}  ({purity*100:.1f}%)')
else:
    purity = 0.0
    print('Clustering degenerado — pureza no aplica')

Validación externa: distribución de grupos de tejido por cluster K-Means



+----------+------------------+-----+
|prediction|      TISSUE_GROUP|count|
+----------+------------------+-----+
|         0|Musculoesqueletico|   55|
|         1|    Cardiovascular|   80|
|         1|Musculoesqueletico|   25|
+----------+------------------+-----+

Tabla de contingencia (cluster × grupo de tejido):
TISSUE_GROUP  Cardiovascular  Musculoesqueletico  Total
prediction                                             
0                          0                  55     55
1                         80                  25    105
Total                     80                  80    160

Pureza del clustering: 0.8438  (84.4%)


### 4.4 Gaussian Mixture Model (GMM) como experimento comparativo

El **GMM** es una alternativa probabilística a K-Means. En lugar de asignar cada punto al centroide más cercano (asignación *hard*), el GMM asigna probabilidades de pertenencia a cada componente gaussiana (asignación *soft*).

**Diferencias clave frente a K-Means:**

| Aspecto | K-Means | GMM |
|---------|---------|-----|
| Tipo de asignación | Hard (cluster definitivo) | Soft (probabilidad por cluster) |
| Forma de clusters | Esférica (isótropa) | Elíptica (covarianza libre) |
| Algoritmo de ajuste | Lloyd (iterativo, cerrado) | EM (Expectation-Maximization) |
| Salida del modelo | Centroide por cluster | Media + covarianza por componente |
| Sensibilidad a outliers | Alta | Moderada |

En el contexto de datos RNA-seq, el GMM puede capturar mejor la variabilidad de expresión dentro de cada tipo de tejido, ya que los perfiles de expresión génica suelen seguir distribuciones log-normales.

In [15]:
from pyspark.ml.clustering import GaussianMixture

gmm = GaussianMixture(
    featuresCol='pca_features',
    predictionCol='prediction',
    probabilityCol='probability',
    k=2,
    maxIter=30,
    seed=RANDOM_SEED
)

print('Entrenando GMM (k=2, PCA 50 componentes, maxIter=30)...')
model_gmm = gmm.fit(train_pca)
print('Entrenamiento completado.')
print(f'\nLog-verosimilitud de entrenamiento: {model_gmm.summary.logLikelihood:.4f}')

Entrenando GMM (k=2, PCA 50 componentes, maxIter=30)...


Entrenamiento completado.

Log-verosimilitud de entrenamiento: -22588.9735


In [16]:
preds_gmm = model_gmm.transform(test_pca)

preds_gmm_pd = preds_gmm.select('pca_features', 'prediction', 'TISSUE_GROUP').toPandas()
y_pred_gmm   = preds_gmm_pd['prediction'].values
X_eval_gmm   = np.vstack([v.toArray() for v in preds_gmm_pd['pca_features']])

print('Distribución de clusters GMM en test:')
unique_gmm, counts_gmm = np.unique(y_pred_gmm, return_counts=True)
for cl, cnt in zip(unique_gmm, counts_gmm):
    print(f'  Cluster {cl}: {cnt} muestras')

if len(unique_gmm) >= 2:
    sil_gmm = sk_silhouette(X_eval_gmm, y_pred_gmm, metric='euclidean')
else:
    sil_gmm = -1.0
    print('AVISO: clustering GMM degenerado')

print(f'\nSilhouette score (GMM, k=2): {sil_gmm:.4f}')

print('\nValidación externa: distribución de grupos de tejido por cluster GMM\n')
preds_gmm.groupBy('prediction', 'TISSUE_GROUP').count() \
    .orderBy('prediction', 'TISSUE_GROUP') \
    .show()

contingency_gmm = pd.crosstab(
    preds_gmm_pd['prediction'],
    preds_gmm_pd['TISSUE_GROUP'],
    margins=True, margins_name='Total'
)
print('Tabla de contingencia GMM:')
print(contingency_gmm)

if len(unique_gmm) >= 2:
    purity_gmm = sum(contingency_gmm.iloc[i].drop('Total').max()
                     for i in range(2)) / len(preds_gmm_pd)
else:
    purity_gmm = 0.0
print(f'\nPureza GMM: {purity_gmm:.4f}  ({purity_gmm*100:.1f}%)')

Distribución de clusters GMM en test:
  Cluster 0: 160 muestras
AVISO: clustering GMM degenerado

Silhouette score (GMM, k=2): -1.0000

Validación externa: distribución de grupos de tejido por cluster GMM



+----------+------------------+-----+
|prediction|      TISSUE_GROUP|count|
+----------+------------------+-----+
|         0|    Cardiovascular|   80|
|         0|Musculoesqueletico|   80|
+----------+------------------+-----+

Tabla de contingencia GMM:
TISSUE_GROUP  Cardiovascular  Musculoesqueletico  Total
prediction                                             
0                         80                  80    160
Total                     80                  80    160

Pureza GMM: 0.0000  (0.0%)


In [17]:
print('=' * 55)
print('  Comparación K-Means vs GMM (k=2, PCA 50 componentes)')
print('=' * 55)
print(f'  Silhouette  K-Means : {sil_km:.4f}')
print(f'  Silhouette  GMM     : {sil_gmm:.4f}')
print(f'  Pureza      K-Means : {purity:.4f}  ({purity*100:.1f}%)')
print(f'  Pureza      GMM     : {purity_gmm:.4f}  ({purity_gmm*100:.1f}%)')
print('=' * 55)

  Comparación K-Means vs GMM (k=2, PCA 50 componentes)
  Silhouette  K-Means : 0.1778
  Silhouette  GMM     : -1.0000
  Pureza      K-Means : 0.8438  (84.4%)
  Pureza      GMM     : 0.0000  (0.0%)


---
### 4.5 Interpretación de resultados

#### Resultados obtenidos

| Métrica | K-Means k=2 | GMM k=2 | K-Means k=5 (mejor Silhouette) |
|---------|-------------|---------|--------------------------------|
| **Silhouette** | 0.1778 | -1.0 (degenerado) | 0.3688 |
| **Pureza** | 84.4% | N/A | — |
| **WCSS (train)** | 254,287 | — | 129,283 |

#### Análisis del método del codo

El Silhouette máximo se obtuvo con **k=5** (0.37), no con k=2 como se esperaría si los tejidos formaran dos grupos perfectamente compactos. Esto revela que el espacio PCA de expresión génica tiene **5 grupos naturales**, probablemente correspondientes a los sub-tipos de tejido:

- Cardiovascular: corazón + vasos sanguíneos (2 sub-clusters)
- Musculoesquelético: músculo, tejido adiposo, piel (3 sub-clusters con perfiles muy distintos)

Esto es coherente con la biología: el perfil de expresión de piel difiere sustancialmente del de músculo esquelético, aunque ambos sean 'musculoesqueléticos' a nivel de grupo de tejido (SMTS).

#### K-Means k=2: estructura parcial descubierta sin etiquetas

Con k=2, K-Means identificó:

- **Cluster 0** (55 muestras): **100% Musculoesquelético** — cluster 'puro' de alta coherencia. Las 55 muestras son las más distintas del espacio PCA (probablemente piel/adiposo con expresión muy diferente a cardiovascular).
- **Cluster 1** (105 muestras): mezcla — 80 Cardiovascular (76%) + 25 Musculoesquelético (24%). Las 25 muestras musculoesqueléticas que caen en este cluster tienen perfiles similares a las cardiovasculares (probablemente músculo cardíaco-adyacente o músculo con alta expresión mitocondrial).

La **pureza del 84.4%** (135/160 muestras correctamente agrupadas con la etiqueta mayoritaria) confirma que K-Means captura **parcialmente** la estructura biológica sin ver las etiquetas.

El **Silhouette de 0.178** es bajo, indicando que los clusters se solapan en el espacio PCA. Esto contrasta con la Tarea 3 donde Random Forest alcanzó 100% de precisión: RF puede explotar los 500 genes de forma no lineal, mientras K-Means solo usa distancias euclidianas en el espacio PCA, que comprime información.

#### GMM k=2: convergencia degenerada

El GMM asignó **todas las 160 muestras de prueba al cluster 0** (solución degenerada). Causas posibles:

1. **Convergencia a óptimo local**: el algoritmo EM puede quedar atrapado en un mínimo local donde un componente tiene peso ≈ 1 y el otro ≈ 0.
2. **Problema de escala en el espacio PCA**: los 50 componentes PCA tienen varianzas muy diferentes (los últimos PCs tienen varianza casi cero), lo que puede desestabilizar la estimación de la covarianza gaussiana.
3. **Alta dimensionalidad relativa** (50 dimensiones para 639 puntos de entrenamiento): con pocas muestras por dimensión, las matrices de covarianza de GMM son difíciles de estimar correctamente.

**Posibles soluciones**: usar `GaussianMixture` con regularización de covarianza (`tol` menor), inicialización con K-Means (`initMode='kmeans'` en PySpark), o reducir los componentes PCA a 10-20.

#### Coherencia con Tarea 3

| Aspecto | Tarea 3 (Supervisado, RF) | Tarea 4 (No supervisado, K-Means) |
|---------|---------------------------|------------------------------------|
| AUC-ROC / Silhouette | 1.0 | 0.18 |
| Precisión / Pureza | 100% | 84.4% |
| Usa etiquetas | Sí | No |
| Información aprovechada | 500 genes + relación no lineal | Distancia euclidiana en PCA-50 |

La diferencia en rendimiento es esperada y explica el valor del aprendizaje supervisado: con etiquetas, el modelo puede aprender las combinaciones exactas de genes que discriminan los tejidos. Sin etiquetas, K-Means solo puede aproximar la estructura geométrica del espacio de expresión, que no es perfectamente bipartito.

#### Supuestos y limitaciones

1. **Independencia entre muestras**: un mismo donante puede contribuir muestras de ambos tejidos — correlación intra-donante entre train y test.
2. **PCA lineal**: captura el 90.7% de la varianza pero descarta relaciones no lineales importantes para clustering.
3. **K-Means asume clusters esféricos**: los sub-tipos de tejido (piel vs músculo) pueden tener formas no esféricas en el espacio PCA.
4. **Selección de genes por varianza** (sin etiquetas): aunque biológicamente justificada, favorece genes con alta varianza que no necesariamente son los mejores para clustering.

---
## Referencias

1. MacQueen, J. B. (1967). Some methods for classification and analysis of multivariate observations. *Proceedings of the 5th Berkeley Symposium on Mathematical Statistics and Probability*, 1, 281–297.
2. Arthur, D., & Vassilvitskii, S. (2007). K-Means++: The advantages of careful seeding. *Proceedings of ACM-SIAM SODA*, 1027–1035.
3. Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society: Series B*, 39(1), 1–38.
4. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
5. Apache Spark MLlib. (2024). Clustering. https://spark.apache.org/docs/latest/ml-clustering.html
6. Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65.

---

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Sonnet 4.6* [Modelo de lenguaje grande], utilizado para soporte en estructura del notebook, documentación de celdas markdown y revisión del código PySpark. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en el autor. Las decisiones de diseño del experimento, selección de algoritmos, supuestos y la interpretación de resultados son del autor.*